# Clip SEVIRI usate nel pretraining VideoMAE

Il notebook legge direttamente dagli Zarr le 100 clip casuali e validate del manifest, senza etichette EMMA/GPM/RDT. La prima riga mostra lo stesso istante nei sette canali; la seconda mostra quattro differenze spettrali diagnostiche. **Play** avanza lungo i 16 frame RSS a cinque minuti.

In [1]:
from pathlib import Path
from functools import lru_cache
import os
import sys

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from emma_gpm.seviri import THERMAL_CHANNELS
from emma_gpm.seviri_clips import SeviriZarrClipReader

ZARR_ROOT = Path(os.environ.get(
    'SEVIRI_ZARR_ROOT',
    r'E:\Datasets\Deep_convection\processed\msg_seviri\rss_7ch_bt_emma_30N70N',
))
MANIFEST = ROOT / 'catalogs' / 'seviri_pretraining_random_100.csv'
manifest = pd.read_csv(MANIFEST)
print(f'Zarr: {ZARR_ROOT}')
print(f'Clip disponibili: {len(manifest)}')
display(manifest)

Zarr: E:\Datasets\Deep_convection\processed\msg_seviri\rss_7ch_bt_emma_30N70N
Clip disponibili: 100


,clip_id,start_time,origin_y,origin_x,selection,seed
0,random_000,2020-05-31T15:00:00Z,211,159,uniform_random_complete_window_and_crop,20261001
1,random_001,2020-08-23T00:25:00Z,475,1011,uniform_random_complete_window_and_crop,20261001
2,random_002,2020-05-29T22:40:00Z,396,791,uniform_random_complete_window_and_crop,20261001
3,random_003,2020-09-17T13:20:00Z,382,221,uniform_random_complete_window_and_crop,20261001
4,random_004,2020-07-13T20:55:00Z,470,599,uniform_random_complete_window_and_crop,20261001
...,...,...,...,...,...,...
95,random_095,2020-06-08T16:00:00Z,366,769,uniform_random_complete_window_and_crop,20261001
96,random_096,2020-09-17T14:15:00Z,274,1354,uniform_random_complete_window_and_crop,20261001
97,random_097,2020-08-15T00:50:00Z,21,455,uniform_random_complete_window_and_crop,20261001
98,random_098,2020-08-21T09:55:00Z,488,1138,uniform_random_complete_window_and_crop,20261001


In [2]:
reader = SeviriZarrClipReader(ZARR_ROOT)
@lru_cache(maxsize=4)
def load_clip(clip_id):
    row = manifest.loc[manifest.clip_id == clip_id].iloc[0]
    return reader.load(
        row.start_time,
        origin_y=int(row.origin_y),
        origin_x=int(row.origin_x),
    )

channel_labels = [
    'WV 6.2 µm', 'WV 7.3 µm', 'IR 8.7 µm', 'IR 9.7 µm',
    'IR 10.8 µm', 'IR 12.0 µm', 'IR 13.4 µm',
]
BT_MIN_K, BT_MAX_K = 180.0, 330.0
spectral_differences = [
    ('WV 6.2 − IR 10.8', 0, 4, -80.0, 20.0),
    ('IR 8.7 − IR 10.8', 2, 4, -15.0, 15.0),
    ('IR 10.8 − IR 12.0', 4, 5, -10.0, 10.0),
    ('IR 10.8 − IR 13.4', 4, 6, -20.0, 40.0),
]
clip_selector = widgets.Dropdown(
    options=manifest.clip_id.tolist(), description='Clip:', layout=widgets.Layout(width='320px')
)
play = widgets.Play(
    value=0, min=0, max=15, step=1, interval=550, description='Play'
)
frame_slider = widgets.IntSlider(
    value=0, min=0, max=15, step=1, description='Frame:', continuous_update=False,
    layout=widgets.Layout(width='520px')
)
widgets.jslink((play, 'value'), (frame_slider, 'value'))
output = widgets.Output()

def render(*_):
    clip_id = clip_selector.value
    frame = frame_slider.value
    clip = load_clip(clip_id)
    row = manifest.loc[manifest.clip_id == clip_id].iloc[0]
    timestamp = pd.Timestamp(row.start_time) + pd.Timedelta(minutes=5 * frame)
    with output:
        clear_output(wait=True)
        fig = plt.figure(figsize=(23, 8.5), constrained_layout=True)
        outer = fig.add_gridspec(2, 1)
        top = outer[0].subgridspec(1, 7)
        bottom = outer[1].subgridspec(1, 4)
        axes = [fig.add_subplot(top[0, index]) for index in range(7)]
        difference_axes = [fig.add_subplot(bottom[0, index]) for index in range(4)]
        for channel, (axis, label) in enumerate(zip(axes, channel_labels)):
            image = axis.imshow(
                clip[frame, channel], cmap='turbo_r', vmin=BT_MIN_K, vmax=BT_MAX_K
            )
            axis.set_title(label, fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
            fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label='K')
        for axis, (label, minuend, subtrahend, vmin, vmax) in zip(
            difference_axes, spectral_differences
        ):
            difference = clip[frame, minuend] - clip[frame, subtrahend]
            image = axis.imshow(
                difference, cmap='RdBu_r',
                norm=TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax),
            )
            axis.set_title(label, fontsize=11)
            axis.set_xticks([])
            axis.set_yticks([])
            fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03, label='ΔK')
        fig.suptitle(f'{clip_id} — {timestamp:%Y-%m-%d %H:%M UTC} — frame {frame + 1}/16', fontsize=15)
        display(fig)
        plt.close(fig)

frame_slider.observe(render, names='value')
clip_selector.observe(lambda change: (setattr(frame_slider, 'value', 0), render()), names='value')
display(widgets.HBox([clip_selector, play, frame_slider]), output)
render()

Output()

C:\Users\Daniele.LOKI\AppData\Local\Temp\ipykernel_4460\2229587352.py:40: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  timestamp = pd.Timestamp(row.start_time) + pd.Timedelta(minutes=5 * frame)


La prima riga usa la stessa scala assoluta e fissa (180–330 K) per tutti i canali, tutte le clip e tutti i frame. La seconda riga mostra differenze spettrali diagnostiche con intervalli fisici prefissati e centrati sullo zero; queste differenze sono derivate per l'ispezione e non vengono aggiunte all'input VideoMAE.

## Sequenza sull'intero dominio mediterraneo ed europeo

I video seguenti mostrano una sola sequenza RSS completa sul dominio **20°W–40°E, 30–60°N**. I limiti longitudinali coincidono con il dominio EMMA usato nel progetto. Le sette bande e le quattro differenze spettrali sono file separati, ma condividono gli stessi 16 istanti a cadenza di cinque minuti.

In [3]:
from IPython.display import Video

MEDITERRANEAN_CLIP_ID = 'random_000'
MEDITERRANEAN_VIDEO_DIR = (
    ROOT / 'figures' / 'videomae_pretraining' / f'mediterranean_{MEDITERRANEAN_CLIP_ID}'
)
mediterranean_index = pd.read_csv(MEDITERRANEAN_VIDEO_DIR / 'video_index.csv')
missing_videos = [
    name for name in mediterranean_index.path
    if not (MEDITERRANEAN_VIDEO_DIR / name).exists()
]
if missing_videos:
    raise FileNotFoundError(f'Video mancanti: {missing_videos}')
display(mediterranean_index[['title', 'path', 'vmin_k', 'vmax_k']])

,title,path,vmin_k,vmax_k
0,WV 6.2 µm,01_WV_062.mp4,180.0,330.0
1,WV 7.3 µm,02_WV_073.mp4,180.0,330.0
2,IR 8.7 µm,03_IR_087.mp4,180.0,330.0
3,IR 9.7 µm,04_IR_097.mp4,180.0,330.0
4,IR 10.8 µm,05_IR_108.mp4,180.0,330.0
5,IR 12.0 µm,06_IR_120.mp4,180.0,330.0
6,IR 13.4 µm,07_IR_134.mp4,180.0,330.0
7,WV 6.2 − IR 10.8,08_WV_062_minus_IR_108.mp4,-80.0,20.0
8,IR 8.7 − IR 10.8,09_IR_087_minus_IR_108.mp4,-15.0,15.0
9,IR 10.8 − IR 12.0,10_IR_108_minus_IR_120.mp4,-10.0,10.0


In [ ]:
mediterranean_selector = widgets.Dropdown(
    options=[(row.title, row.path) for row in mediterranean_index.itertuples()],
    description='Video:',
    layout=widgets.Layout(width='520px'),
)
mediterranean_output = widgets.Output()

def show_mediterranean_video(*_):
    with mediterranean_output:
        clear_output(wait=True)
        path = MEDITERRANEAN_VIDEO_DIR / mediterranean_selector.value
        display(Video(filename=str(path), embed=False, width=1050, html_attributes='controls loop'))

mediterranean_selector.observe(show_mediterranean_video, names='value')
display(mediterranean_selector, mediterranean_output)
show_mediterranean_video()

Dropdown(description='Video:', layout=Layout(width='520px'), options=(('WV 6.2 µm', '01_WV_062.mp4'), ('WV 7.3…

Output()

In [5]:
# Funzione opzionale per produrre gli undici video per un'altra clip del manifest.
import subprocess

def render_mediterranean_videos(clip_id):
    command = [
        sys.executable,
        str(ROOT / 'scripts' / 'render_seviri_mediterranean_videos.py'),
        '--manifest', str(MANIFEST),
        '--clip-id', clip_id,
    ]
    subprocess.run(command, cwd=ROOT, check=True)
    return ROOT / 'figures' / 'videomae_pretraining' / f'mediterranean_{clip_id}'

# Esempio (richiede alcuni minuti):
# render_mediterranean_videos('random_001')